In [ ]:
import jax.numpy as jnp
import numpy as np
from fft import *
import jax
from symmetry import *
from phase_retrieval import *
from obj import *
import sys
from noise import *
from mle import *
from plotting import plot
import device
from jax import lax
from pies import *
from pies import *
from symmetry import choose_representative
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", False)
gpu_idx, free_mb, gpu_count = device.setup_device(device_id=1)

From CDI with oversampled diffraction patterns to real-space ptychography
- No need for oversampling or 'padding' in implementation
- Additional axis: illumination
    - Initially: uniform square or circle?
    - Eventually: high-NA spot with finite frequency support rather than object support  

In [ ]:
N_crop = 255

N_pad = 0
N_iter_HIO = 1000
beta_HIO = 0.99
epsilon_ML = 1e-12
N_iter_ML = 1000

real_object = False

# Ensure that cameraman is nowhere exactly zero to avoid possible singular behavior (for now)
nonzero_cameraman = cameraman() * 0.9 + 0.1

if real_object:
    # Construct object with cameraman in intensity
    x_obj, obj_support = crop_and_pad(nonzero_cameraman, N_crop=N_crop, N_pad=N_pad)
else:
    # Construct object with cameraman in intensity and mandrill in phase
    x_cameraman = nonzero_cameraman
    x_mandrill = as_phase_field(mandrill() ** 2 - 0.5)
    x_obj, obj_support = crop_and_pad(
        x_cameraman * x_mandrill, N_crop=N_crop, N_pad=N_pad
    )

x, y, X, Y, R, k_0, k_x, k_y, K_X, K_Y, k, r_extent, k_extent = create_physical_axes(
    x_obj
)

N_compare_crop = N_crop // 2


def compare_estimate(x_est_raw, x_obj, N_compare_crop=N_compare_crop):
    # Break phase and if applicable twin symmetry
    x_est = choose_representative(x_est_raw, x_obj)
    # Compensate seemingly common global amplitude scale mismatches
    obj_crop, _ = crop_and_pad(x_obj, N_compare_crop, 0)
    est_crop, _ = crop_and_pad(x_est, N_compare_crop, 0)

    est_crop_norm = normalize_image_intensity(est_crop)
    obj_crop_norm = normalize_image_intensity(obj_crop)
    plot_intensity_phase(obj_crop_norm)
    plot_intensity_phase(est_crop_norm)


print(r_extent, k_extent)
# Define linear operator
A = ifft2_shifted
A_dagger = fft2_shifted
yhat = A(x_obj)
plot_intensity_phase(x_obj, extent=r_extent)
plot_intensity_phase(yhat, extent=k_extent, use_log=True)

# 1. Real-space ptychography with multi-spot array probe
As a first step, the probe pattern is created.

In [ ]:
from functools import partial
from spotgen import *

min_x, max_x, min_y, max_y = r_extent
# Fraction of r_extent to fill with the multispot array
frac_x = 0.25
NA = 0.5

N_x = 5
N_y = 5
base_probe = create_multispot_array_product(
    NA,
    min_x * frac_x,
    max_x * frac_x,
    min_y * frac_x,
    max_y * frac_x,
    N_x,
    N_y,
    K_X,
    K_Y,
    k,
    k_0,
)
base_probe_fast = create_multispot_array_implicit(
    NA,
    min_x * frac_x,
    max_x * frac_x,
    min_y * frac_x,
    max_y * frac_x,
    N_x,
    N_y,
    k_x,
    k_y,
    k,
    k_0,
)
plot_intensity_phase(base_probe, extent=r_extent)
plot_intensity_phase(ifft2_shifted(base_probe), extent=k_extent)
plot_intensity_phase(base_probe - base_probe_fast)

In [ ]:
# %timeit create_multispot_array_original(NA, min_x * frac_x, max_x * frac_x, min_y * frac_x, max_y * frac_x, 100, 100).block_until_ready()
# %timeit create_multispot_array_loops(NA, min_x * frac_x, max_x * frac_x, min_y * frac_x, max_y * frac_x, 100, 100).block_until_ready()
%timeit create_multispot_array_explicit(NA, min_x * frac_x, max_x * frac_x, min_y * frac_x, max_y * frac_x, 100, 100,K_X,K_Y,k,k_0).block_until_ready()
%timeit create_multispot_array_product(NA, min_x * frac_x, max_x * frac_x, min_y * frac_x, max_y * frac_x, 100, 100,K_X,K_Y,k,k_0).block_until_ready()
%timeit create_multispot_array_implicit(NA, min_x * frac_x, max_x * frac_x, min_y * frac_x, max_y * frac_x, 100, 100,k_x,k_y,k,k_0).block_until_ready()

In [ ]:
frac_x = 0.5
base_probe = create_multispot_array_implicit(
    0.25,
    min_x * frac_x,
    max_x * frac_x,
    min_y * frac_x,
    max_y * frac_x,
    16,
    8,
    k_x,
    k_y,
    k,
    k_0,
)
N_photons = 1e12
probe = normalize_image_intensity(base_probe) * jnp.sqrt(N_photons)
plot_intensity_phase(probe, extent=r_extent)
plot_intensity_phase(ifft2_shifted(probe), extent=k_extent)

In [ ]:
x = x_obj * probe
plot_intensity_phase(x)
plot_intensity_phase(fft2_shifted(x), use_log=True)

In [ ]:
pupil = create_displaced_spot_spectrum(0.5, 0, 0, K_X, K_Y, k, k_0)
x_prime = fft2_shifted(ifft2_shifted(x_obj) * pupil)
plot_intensity_phase(x_prime)

# 1.1. Scanning grid equations
For pitch $p$, array size (counts) in $x$-direction $L_x$ and scanning angle $\gamma$, these must hold.
$$p \sin \gamma = R/2$$
$$p \cos \gamma = L_xR/2$$
$$\implies \tan \gamma = 1/L_x$$

In [ ]:
L_x, L_y = 22, 64
# L_x, L_y = 20, 64 also gives something reasonable, it has vertical artefacts though, perpendicular to (fine) scanning direction.
NA = 0.5
R_opt = 1 / (2 * NA)  # lambda
resolution = R_opt
print(R_opt)
frac_x = 0.3
frac_y = 0.9
array_bounds = min_x * frac_x, max_x * frac_x, min_y * frac_y, max_y * frac_y
p_x = (array_bounds[1] - array_bounds[0]) / (L_x - 1)
p_y = (array_bounds[3] - array_bounds[2]) / (L_y - 1)
print(p_x, p_y, resolution)
# gamma = np.arccos(L_x * R / (2 * p))
# print(L_x * R / (2 * p))
angle_gamma = np.arctan(1 / L_x)

# 1.2. Simulate $M$ illuminations of sample by shifted multi-spot probe

In [ ]:
M = 200
shift_xs = jnp.linspace(-20 * p_x, 20 * p_x, M)
cdtype = jnp.array(1j).dtype
print(cdtype)
R_probes = jnp.zeros((M, 2))
probes = jnp.zeros((M, *k.shape), dtype=cdtype)

for m, shift_diag in tqdm(enumerate(shift_xs)):
    R_x = np.cos(angle_gamma) * shift_diag
    R_y = np.sin(angle_gamma) * shift_diag
    R_probes = R_probes.at[m, :].set(jnp.array([R_x, R_y]))
    base_probe = create_multispot_array_product(
        NA,
        array_bounds[0] + R_x,
        array_bounds[1] + R_x,
        array_bounds[2] + R_y,
        array_bounds[3] + R_y,
        L_x,
        L_y,
        K_X,
        K_Y,
        k,
        k_0,
        # k_x,
        # k_y,
    )
    N_photons = 1e12
    probe = normalize_image_intensity(base_probe) * jnp.sqrt(N_photons)
    probes = probes.at[m, :].set(probe)

ys = fft2_shifted(x_obj * probes)
plot_intensity_phase(ys[-1, :], use_log=True)

probe_sum = jnp.sum(jnp.abs(probes) ** 2, axis=0)
object_probe_sum = jnp.sum(jnp.abs(probes * x_obj) ** 2, axis=0)
probed = jnp.astype(probe_sum >= jnp.max(probe_sum) * 1e-2, jnp.float64)
plot(probe_sum, r_extent)
plot(object_probe_sum, r_extent)
plot(jnp.abs(probes[M // 2, :]), r_extent)

In [ ]:
plt.scatter(R_probes[:, 0] / p_x, R_probes[:, 1] / p_x)
plt.xlabel(r"$R_{m,x}$ [$p_x$]")
plt.ylabel(r"$R_{m,y}$ [$p_x$]")

# 1.3. Ptychographical Iterative Engine (PIE) reconstruction

In [ ]:
x_est = ppie(ys, probes, gamma=0.75, N_iter=1000)
compare_estimate(x_est * probed, x_obj * probed)

# 2. Basic real-space ptychography with circular probe and 2D scanning

The eventual goal is to find a way to scan 1D and fully reconstruct the object. First, consider the traditional real-space ptychography setting with 2D scanning and a circular probe.

The area overlap fraction for circles with radius $R$ a distance $r$ apart is

$$f(r,R) = \frac{2}{\pi}\arccos(\frac{r}{2R}) - \frac{r}{2\pi R^2}\sqrt{4R^2 - r^2}, 0\leq r \leq 2R$$

In [ ]:
import matplotlib.pyplot as plt

R_spot = 10  # lambda

N_fs = 1000
rs = jnp.linspace(0, 2 * R_spot, N_fs)
f_r = 2 / jnp.pi * jnp.arccos(rs / (2 * R_spot)) - rs / (
    2 * jnp.pi * R_spot**2
) * jnp.sqrt(4 * R_spot**2 - rs**2)
# plt.plot(r,f_r)

overlap_fraction = 0.65
f = 1.0

# Could be more efficient... Oh well
for i in range(N_fs):
    f = f_r[i]
    if f <= overlap_fraction:
        r = rs[i]
        break
# Happens to be almost exactly 2
r = round(r)
r

In [ ]:
def shift(x, delta_x, delta_y):
    x_spectrum = fft2_shifted(x)
    displacement_ramp = jnp.exp(-1.0j * (delta_x * K_X + delta_y * K_Y))
    shifted_x_spectrum = x_spectrum * displacement_ramp
    shifted_x = ifft2_shifted(shifted_x_spectrum)
    return shifted_x


def create_shifted_spot(R_spot, delta_x, delta_y):
    spot = jnp.astype(R <= R_spot, jnp.float64)
    shifted_spot = shift(spot, delta_x, delta_y)
    return shifted_spot


shifted_spot = create_shifted_spot(R_spot, 0, 0)
plot(jnp.abs(shifted_spot), extent=r_extent)
plot_intensity_phase(ifft2_shifted(shifted_spot), extent=r_extent)

In [ ]:
M_x = 10
M_y = 10
M = M_x * M_y
base_key = jax.random.PRNGKey(1776)
delta_r = r * 0.2
deltas = jax.random.uniform(base_key, shape=(M, 2), minval=-delta_r, maxval=delta_r)


probes = jnp.zeros((M, *k.shape), dtype=cdtype)
R_probes = jnp.zeros((M, 2))
for i in range(M_x):
    for j in range(M_y):
        m = i * M_x + j
        R_x = (i - (M_x - 1) / 2) * r + deltas[m, 0]
        R_y = (j - (M_y - 1) / 2) * r + deltas[m, 1]
        R_probes = R_probes.at[m, :].set(jnp.array([R_x, R_y]))

        base_probe = create_shifted_spot(
            R_spot,
            R_x,
            R_y,
        )
        N_photons = 1
        probe = normalize_image_intensity(base_probe) * jnp.sqrt(N_photons)
        probes = probes.at[m, :].set(probe)

ys = fft2_shifted(x_obj * probes)
plot_intensity_phase(ys[-1, :], use_log=True)


probe_sum = jnp.sum(jnp.abs(probes) ** 2, axis=0)
object_probe_sum = jnp.sum(jnp.abs(probes * x_obj) ** 2, axis=0)
probed = jnp.astype(probe_sum >= jnp.max(probe_sum) * 1e-2, jnp.float64)
plot(probe_sum, r_extent)
plot(object_probe_sum, r_extent)
plot(jnp.real(probes[-1, :]), r_extent)
plot(probed, r_extent)

In [ ]:
plt.scatter(R_probes[:, 0] / r, R_probes[:, 1] / r)
plt.xlabel(r"$R_{m,x}$ [p]")
plt.ylabel(r"$R_{m,y}$ [p]")

In [ ]:
x_est_serial = pie(
    ys, probes, alpha=0.5, gamma=0.75, N_iter=400, obj_support=obj_support
)
compare_estimate(x_est_serial * probed, x_obj * probed)

In [ ]:
x_est = ppie(ys, probes, gamma=0.75, N_iter=400, obj_support=obj_support)
compare_estimate(x_est * probed, x_obj * probed)

In [ ]:
# x_est_4 = ppie_multigpu(
#     ys,
#     probes,
#     gamma=0.75,
#     # base_key=jax.random.PRNGKey(1776),
#     N_gpus=gpu_count,
#     N_iter=400,
#     obj_support=obj_support * probed,
# )
# for i in range(gpu_count):
#     compare_estimate(x_est_4[i, :], x_obj)